<a href="https://colab.research.google.com/github/abstudios-design/animate/blob/main/wan2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/theelderemo/wan2.2-google-colab/blob/main/wan2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wan2.2 — Complete Video Generation Suite

This notebook covers **every generation mode** in Wan2.2:

| # | Mode | Model | Notes |
|---|------|-------|-------|
| 1 | **Text-to-Video** | T2V-A14B (MoE 27B / 14B active) | 480P & 720P |
| 2 | **Image-to-Video** | I2V-A14B (MoE 27B / 14B active) | 480P & 720P |
| 3 | **Text+Image-to-Video** | TI2V-5B | 720P @ 24 FPS, runs on 4090 |
| 4 | **Speech-to-Video** | S2V-14B | Audio + image → talking-head video |
| 4a | **Speech-to-Video + Pose** | S2V-14B | Pose-driven with audio |
| 4b | **Speech-to-Video + TTS** | S2V-14B + CosyVoice | Generate speech then animate |
| 5 | **Character Animation** | Animate-14B | Mimic human motion from video |
| 6 | **Character Replacement** | Animate-14B | Replace character in video |

**Workflow:** Run **Section 0** (setup) once → **Section 1** (download only the models you need) → jump to any generation section.

---
**GPU requirements at a glance:**
- TI2V-5B (Section 3): ≥ 24 GB VRAM (RTX 4090 works)
- All other models: ≥ 80 GB VRAM (A100/H100), or use `offload_model + convert_model_dtype` flags
- Colab A100 recommended for the A14B models

---
## Section 0 — Setup & Installation

Run these cells **once per Colab session**.

In [1]:
!pip install --no-cache-dir --force-reinstall --no-deps "huggingface_hub>=1.32,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 23.7 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0


In [1]:
import huggingface_hub

print("Hugging Face Hub:", huggingface_hub.__version__)

from huggingface_hub import HfApi

api = HfApi()

print("Hugging Face API: Working")

Hugging Face Hub: 1.33.0
Hugging Face API: Working


In [ ]:
# ── 0.1  Check GPU ────────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU detected — check Colab runtime type.")

In [ ]:
# ── 0.2  Clone Wan2.2 repo ────────────────────────────────────────────────────
import os

if not os.path.isdir("Wan2.2"):
    !git clone https://github.com/Wan-Video/Wan2.2.git
else:
    print("Wan2.2 directory already exists — skipping clone.")
    !cd Wan2.2 && git pull

In [ ]:
# ── 0.3  Install core dependencies ───────────────────────────────────────────
%cd Wan2.2
!pip install -q -r requirements.txt
!pip install -q .
!pip install -q decord
print("Core dependencies installed.")

In [2]:
# ── 0.4  Install flash-attn (optional but recommended for speed) ──────────────
# This step can take 5-10 minutes. Skip if you hit issues — the model will
# fall back to standard attention automatically.
!pip install -q flash-attn --no-build-isolation
print("flash-attn installed.")

  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for flash-attn (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> flash-attn
flash-attn installed.


In [ ]:
# ── 0.5  Install S2V / CosyVoice dependencies (only needed for Section 4b) ───
# @markdown Run this cell only if you plan to use Speech-to-Video with TTS.
install_s2v = True  # @param {type:"boolean"}

if install_s2v:
    !pip install -q -r requirements_s2v.txt
    print("S2V/CosyVoice dependencies installed.")
else:
    print("Skipped S2V dependencies.")

In [ ]:
# ── 0.6  Install huggingface-hub CLI ─────────────────────────────────────────
!pip install -q "huggingface_hub[cli]"

# Optional: log in to HF to avoid rate limits on large model downloads
# from huggingface_hub import login
# login()   # uncomment and run if you have an HF token
print("huggingface_hub CLI ready.")

---
## Section 1 — Model Download

Download **only the model(s) you need**. Each model cell is independent.

| Model | Size (approx) | Use case |
|-------|--------------|----------|
| T2V-A14B | ~28 GB | Text → Video (480P / 720P) |
| I2V-A14B | ~28 GB | Image → Video (480P / 720P) |
| TI2V-5B  | ~10 GB | Text+Image → Video 720P@24fps, 4090-friendly |
| S2V-14B  | ~28 GB | Speech/Audio → Video |
| Animate-14B | ~28 GB | Character animation & replacement |

In [ ]:
# ── 1.1  Text-to-Video model — T2V-A14B ──────────────────────────────────────
import os
if not os.path.isdir("./Wan2.2-T2V-A14B"):
    !huggingface-cli download Wan-AI/Wan2.2-T2V-A14B --local-dir ./Wan2.2-T2V-A14B
else:
    print("T2V-A14B already downloaded.")

In [ ]:
# ── 1.2  Image-to-Video model — I2V-A14B ─────────────────────────────────────
import os
if not os.path.isdir("./Wan2.2-I2V-A14B"):
    !huggingface-cli download Wan-AI/Wan2.2-I2V-A14B --local-dir ./Wan2.2-I2V-A14B
else:
    print("I2V-A14B already downloaded.")

In [ ]:
# ── 1.3  Text+Image-to-Video model — TI2V-5B ─────────────────────────────────
# Smaller and faster — great for consumer GPUs (24 GB+)
import os
if not os.path.isdir("./Wan2.2-TI2V-5B"):
    !huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B
else:
    print("TI2V-5B already downloaded.")

In [ ]:
# ── 1.4  Speech-to-Video model — S2V-14B ─────────────────────────────────────
import os
if not os.path.isdir("./Wan2.2-S2V-14B"):
    !huggingface-cli download Wan-AI/Wan2.2-S2V-14B --local-dir ./Wan2.2-S2V-14B
else:
    print("S2V-14B already downloaded.")

In [ ]:
# ── 1.5  Character Animation model — Animate-14B ─────────────────────────────
import os
if not os.path.isdir("./Wan2.2-Animate-14B"):
    !huggingface-cli download Wan-AI/Wan2.2-Animate-14B --local-dir ./Wan2.2-Animate-14B
else:
    print("Animate-14B already downloaded.")

---
## Section 2 — Text-to-Video (T2V-A14B)

Generates video purely from a text prompt using the MoE 27B model (14B active parameters).
Supports **480P** and **720P** output.

In [ ]:
# ── 2.1  T2V Configuration ────────────────────────────────────────────────────
# @title Text-to-Video Settings

# @markdown ### Prompt
t2v_prompt = "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage."  # @param {type:"string"}

# @markdown ### Resolution
t2v_resolution = "1280*720"  # @param ["1280*720", "832*480"]

# @markdown ### Inference Steps & Seed
t2v_steps = 50  # @param {type:"integer"}
t2v_seed = 42   # @param {type:"integer"}

# @markdown ### Memory Optimization (enable if you hit OOM)
t2v_offload = True        # @param {type:"boolean"}
t2v_convert_dtype = True  # @param {type:"boolean"}
t2v_t5_cpu = False        # @param {type:"boolean"}

# @markdown ### Prompt Extension
# @markdown Leave `use_prompt_extend` off for fastest run. Enable to enrich prompt details.
t2v_use_prompt_extend = False     # @param {type:"boolean"}
t2v_extend_method = "local_qwen" # @param ["local_qwen", "dashscope"]
t2v_extend_model  = "Qwen/Qwen2.5-7B-Instruct"  # @param {type:"string"}
t2v_dashscope_key = ""           # @param {type:"string"}

# Build args string
t2v_args = ""
if t2v_offload:       t2v_args += "--offload_model True "
if t2v_convert_dtype: t2v_args += "--convert_model_dtype "
if t2v_t5_cpu:        t2v_args += "--t5_cpu "
if t2v_use_prompt_extend:
    t2v_args += f"--use_prompt_extend --prompt_extend_method '{t2v_extend_method}' "
    if t2v_extend_method == "local_qwen":
        t2v_args += f"--prompt_extend_model '{t2v_extend_model}' "

print("T2V configuration ready.")
print(f"  Resolution : {t2v_resolution}")
print(f"  Steps      : {t2v_steps}")
print(f"  Extra args : {t2v_args.strip()}")

In [ ]:
# ── 2.2  Run Text-to-Video Generation ────────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

_env = f"DASH_API_KEY={t2v_dashscope_key} " if t2v_dashscope_key else ""

!{_env}python generate.py \
    --task t2v-A14B \
    --size {t2v_resolution} \
    --ckpt_dir ./Wan2.2-T2V-A14B \
    --sample_steps {t2v_steps} \
    --base_seed {t2v_seed} \
    --prompt "{t2v_prompt}" \
    {t2v_args}

---
## Section 3 — Image-to-Video (I2V-A14B)

Animates a still image into a video. Supports **480P** and **720P**.
The output aspect ratio follows the input image's aspect ratio.

In [ ]:
# ── 3.1  Upload Input Image ───────────────────────────────────────────────────
from google.colab import files
import os

print("Upload the image you want to animate:")
uploaded = files.upload()

if uploaded:
    i2v_input_image = list(uploaded.keys())[0]
    print(f"Image set to: '{i2v_input_image}'")
else:
    # Fall back to the bundled example
    i2v_input_image = "examples/i2v_input.JPG"
    print(f"No upload detected — using example image: {i2v_input_image}")

In [ ]:
# ── 3.2  I2V Configuration ────────────────────────────────────────────────────
# @title Image-to-Video Settings

# @markdown ### Prompt  (leave empty to auto-generate from the image via prompt extension)
i2v_prompt = "Summer beach vacation style, a white cat wearing sunglasses sits on a surfboard, relaxed expression, crystal-clear waters in the background."  # @param {type:"string"}

# @markdown ### Resolution (area — aspect ratio follows the image)
i2v_resolution = "1280*720"  # @param ["1280*720", "832*480"]

# @markdown ### Inference Steps & Seed
i2v_steps = 50  # @param {type:"integer"}
i2v_seed  = 42  # @param {type:"integer"}

# @markdown ### Memory Optimization
i2v_offload       = True   # @param {type:"boolean"}
i2v_convert_dtype = True   # @param {type:"boolean"}
i2v_t5_cpu        = False  # @param {type:"boolean"}

# @markdown ### Prompt Extension  (auto-describes image when prompt is empty)
i2v_use_prompt_extend = False       # @param {type:"boolean"}
i2v_extend_method     = "local_qwen" # @param ["local_qwen", "dashscope"]
i2v_extend_model      = "Qwen/Qwen2.5-VL-7B-Instruct"  # @param {type:"string"}
i2v_dashscope_key     = ""          # @param {type:"string"}

i2v_args = ""
if i2v_offload:       i2v_args += "--offload_model True "
if i2v_convert_dtype: i2v_args += "--convert_model_dtype "
if i2v_t5_cpu:        i2v_args += "--t5_cpu "
if i2v_use_prompt_extend:
    i2v_args += f"--use_prompt_extend --prompt_extend_method '{i2v_extend_method}' "
    if i2v_extend_method == "local_qwen":
        i2v_args += f"--prompt_extend_model '{i2v_extend_model}' "

print("I2V configuration ready.")

In [ ]:
# ── 3.3  Run Image-to-Video Generation ───────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

if not os.path.exists(i2v_input_image):
    print(f"ERROR: Image '{i2v_input_image}' not found. Run cell 3.1 first.")
else:
    _env = f"DASH_API_KEY={i2v_dashscope_key} " if i2v_dashscope_key else ""
    !{_env}python generate.py \
        --task i2v-A14B \
        --size {i2v_resolution} \
        --ckpt_dir ./Wan2.2-I2V-A14B \
        --image "{i2v_input_image}" \
        --sample_steps {i2v_steps} \
        --base_seed {i2v_seed} \
        --prompt "{i2v_prompt}" \
        {i2v_args}

---
## Section 4 — Text+Image-to-Video (TI2V-5B)

Unified model supporting **both T2V and I2V** from a single 5B checkpoint.  
Uses the high-compression Wan2.2-VAE (16×16×4). Outputs **720P @ 24 FPS**.  
Runs on consumer GPUs with ≥ 24 GB VRAM (e.g. RTX 4090).

> Resolution for this model is `1280*704` (landscape) or `704*1280` (portrait).

In [ ]:
# ── 4.1  Upload Image (optional — omit for pure text-to-video) ───────────────
from google.colab import files
import os

# @markdown Set `use_image` to False for pure Text-to-Video with TI2V-5B.
ti2v_use_image = True  # @param {type:"boolean"}

ti2v_input_image = ""
if ti2v_use_image:
    print("Upload a reference image (or skip to use the bundled example):")
    uploaded = files.upload()
    if uploaded:
        ti2v_input_image = list(uploaded.keys())[0]
        print(f"Image: {ti2v_input_image}")
    else:
        ti2v_input_image = "examples/i2v_input.JPG"
        print(f"Using example: {ti2v_input_image}")
else:
    print("No image — running in Text-to-Video mode.")

In [ ]:
# ── 4.2  TI2V-5B Configuration ────────────────────────────────────────────────
# @title TI2V-5B Settings

ti2v_prompt     = "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage."  # @param {type:"string"}
ti2v_resolution = "1280*704"  # @param ["1280*704", "704*1280"]
ti2v_steps      = 50   # @param {type:"integer"}
ti2v_seed       = 42   # @param {type:"integer"}

# @markdown ### Memory Optimization (24 GB GPU: keep all True)
ti2v_offload       = True  # @param {type:"boolean"}
ti2v_convert_dtype = True  # @param {type:"boolean"}
ti2v_t5_cpu        = True  # @param {type:"boolean"}

ti2v_args = ""
if ti2v_offload:       ti2v_args += "--offload_model True "
if ti2v_convert_dtype: ti2v_args += "--convert_model_dtype "
if ti2v_t5_cpu:        ti2v_args += "--t5_cpu "
if ti2v_input_image:   ti2v_args += f'--image "{ti2v_input_image}" '

mode_label = "Image-to-Video" if ti2v_input_image else "Text-to-Video"
print(f"TI2V-5B mode: {mode_label}")

In [ ]:
# ── 4.3  Run TI2V-5B Generation ───────────────────────────────────────────────
import torch
torch.cuda.empty_cache()

!python generate.py \
    --task ti2v-5B \
    --size {ti2v_resolution} \
    --ckpt_dir ./Wan2.2-TI2V-5B \
    --sample_steps {ti2v_steps} \
    --base_seed {ti2v_seed} \
    --prompt "{ti2v_prompt}" \
    {ti2v_args}

---
## Section 5 — Speech-to-Video (S2V-14B)

Generates a talking-head / audio-driven video from:
- A **reference image** (portrait / character)
- An **audio file** (WAV/MP3 — speech, singing, etc.)
- An optional **text prompt**
- An optional **pose video** for full-body motion

Three sub-modes are provided:
- **5a**: Basic Speech-to-Video (image + audio)
- **5b**: Pose-driven Speech-to-Video (image + audio + pose video)
- **5c**: TTS-powered (CosyVoice synthesizes the audio for you)

In [ ]:
# ── 5.1  Upload Image & Audio ─────────────────────────────────────────────────
from google.colab import files
import os

print("── Upload reference image (portrait / character) ──")
up_img = files.upload()
if up_img:
    s2v_image = list(up_img.keys())[0]
else:
    s2v_image = "examples/i2v_input.JPG"
print(f"Image: {s2v_image}")

print("\n── Upload audio file (WAV or MP3) ──")
up_audio = files.upload()
if up_audio:
    s2v_audio = list(up_audio.keys())[0]
else:
    s2v_audio = "examples/talk.wav"
print(f"Audio: {s2v_audio}")

In [ ]:
# ── 5.2a  Basic Speech-to-Video Configuration ─────────────────────────────────
# @title S2V — Basic (image + audio)

s2v_prompt     = "A person speaking expressively."  # @param {type:"string"}
s2v_resolution = "1024*704"  # @param ["1024*704", "832*480", "1280*720"]
s2v_steps      = 40    # @param {type:"integer"}
s2v_seed       = 42    # @param {type:"integer"}

# @markdown `num_clip`: number of video clips (leave 0 to auto-match audio length)
s2v_num_clip   = 0     # @param {type:"integer"}

s2v_offload       = True  # @param {type:"boolean"}
s2v_convert_dtype = True  # @param {type:"boolean"}

s2v_args = ""
if s2v_offload:       s2v_args += "--offload_model True "
if s2v_convert_dtype: s2v_args += "--convert_model_dtype "
if s2v_num_clip > 0:  s2v_args += f"--num_clip {s2v_num_clip} "

print("S2V (basic) configuration ready.")

In [ ]:
# ── 5.2a  Run Basic Speech-to-Video ──────────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

for path, label in [(s2v_image, "image"), (s2v_audio, "audio")]:
    if not os.path.exists(path):
        print(f"ERROR: {label} file '{path}' not found. Upload in cell 5.1.")
        raise SystemExit()

!python generate.py \
    --task s2v-14B \
    --size {s2v_resolution} \
    --ckpt_dir ./Wan2.2-S2V-14B \
    --image "{s2v_image}" \
    --audio "{s2v_audio}" \
    --sample_steps {s2v_steps} \
    --base_seed {s2v_seed} \
    --prompt "{s2v_prompt}" \
    {s2v_args}

In [ ]:
# ── 5.3b  Pose-Driven Speech-to-Video ─────────────────────────────────────────
# @title S2V — Pose-Driven (image + audio + pose video)
# @markdown Upload a **pose video** (.mp4) that provides the body motion sequence.

from google.colab import files

print("Upload pose video (.mp4):")
up_pose = files.upload()
if up_pose:
    s2v_pose_video = list(up_pose.keys())[0]
else:
    s2v_pose_video = "examples/pose.mp4"
print(f"Pose video: {s2v_pose_video}")

# Reuse image/audio from 5.1 or override here:
# s2v_image = "examples/pose.png"
# s2v_audio = "examples/sing.MP3"

s2v_pose_prompt = "a person is singing"  # @param {type:"string"}

In [ ]:
# ── 5.3b  Run Pose-Driven Speech-to-Video ────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

!python generate.py \
    --task s2v-14B \
    --size {s2v_resolution} \
    --ckpt_dir ./Wan2.2-S2V-14B \
    --image "{s2v_image}" \
    --audio "{s2v_audio}" \
    --pose_video "{s2v_pose_video}" \
    --prompt "{s2v_pose_prompt}" \
    --offload_model True \
    --convert_model_dtype

In [ ]:
# ── 5.4c  TTS-powered Speech-to-Video (CosyVoice) ────────────────────────────
# @title S2V — TTS Mode  (requires S2V deps from cell 0.5)
# @markdown CosyVoice synthesizes audio from text, then Wan2.2-S2V animates the character.

# @markdown #### Reference audio for voice cloning (upload a short WAV of the target speaker)
from google.colab import files
print("Upload reference speaker audio for voice cloning:")
up_ref = files.upload()
if up_ref:
    tts_ref_audio = list(up_ref.keys())[0]
else:
    tts_ref_audio = "examples/zero_shot_prompt.wav"
print(f"Reference audio: {tts_ref_audio}")

# @markdown #### Text of the reference clip (transcription of the audio above)
tts_ref_text = "希望你以后能够做的比我还好呦。"  # @param {type:"string"}

# @markdown #### Text to synthesize and animate (the actual speech content)
tts_text = "收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。"  # @param {type:"string"}

tts_prompt = "A person giving an emotional speech."  # @param {type:"string"}

In [ ]:
# ── 5.4c  Run TTS Speech-to-Video ────────────────────────────────────────────
import os, torch
torch.cuda.empty_cache()

!python generate.py \
    --task s2v-14B \
    --size {s2v_resolution} \
    --ckpt_dir ./Wan2.2-S2V-14B \
    --image "{s2v_image}" \
    --enable_tts \
    --tts_prompt_audio "{tts_ref_audio}" \
    --tts_prompt_text "{tts_ref_text}" \
    --tts_text "{tts_text}" \
    --prompt "{tts_prompt}" \
    --offload_model True \
    --convert_model_dtype

---
## Section 6 — Character Animation & Replacement (Animate-14B)

Wan-Animate takes a **reference character image** and a **source motion video**, and either:
- **Animation mode**: generates a new video of the character mimicking the motion.
- **Replacement mode**: replaces the character in the original video.

**Preprocessing is required** before inference. Two sub-sections below.

In [7]:
# ── 6.1  Upload Assets for Animate ───────────────────────────────────────────
from google.colab import files
import os

print("── Upload reference character image (JPEG/PNG) ──")
up_char = files.upload()
animate_ref_image = list(up_char.keys())[0] if up_char else "examples/wan_animate/animate/image.jpeg"
print(f"Character image: {animate_ref_image}")

print("\n── Upload source motion video (MP4) ──")
up_vid = files.upload()
animate_src_video = list(up_vid.keys())[0] if up_vid else "examples/wan_animate/animate/video.mp4"
print(f"Motion video: {animate_src_video}")

── Upload reference character image (JPEG/PNG) ──


Saving madhuri.jpg to madhuri.jpg
Character image: madhuri.jpg

── Upload source motion video (MP4) ──


Saving reel.mp4 to reel.mp4
Motion video: reel.mp4


In [8]:
import os

print("Uploaded files:")

for f in os.listdir("/content/Wan2.2"):
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".mp4", ".mov")):
        print(f)

Uploaded files:
madhuri.jpg
SnapReels.Net_AQMQfAuNZtvdrr9e74anhPa7TzqgS394mVk0Pv2DUVwgUNRGXoAARfd-1iKbqiLh6hXNPUsUK2AwqLq-Mo1d1KLIyRD_6hTi3Rs9y7c.mp4
1b959d27-b855-48cb-9050-29e0b8595811.png
0b7536c6-5832-4dc6-ab17-90e9ee2d1077 (1).png
reel.mp4
video_ComfyUI_00001_ (1).mp4
video_ComfyUI_00001_ (1) (1).mp4
0b7536c6-5832-4dc6-ab17-90e9ee2d1077.png
main-character.jpg


In [9]:
import os

os.chdir("/content/Wan2.2")

animate_mode = "replace"
animate_res_w = 832
animate_res_h = 480

animate_proc_dir = "./animate_proc_replace"

os.makedirs(animate_proc_dir, exist_ok=True)

print("Mode:", animate_mode)
print("Character:", os.path.abspath("madhuri.jpg"))
print("Source video:", os.path.abspath("reel.mp4"))
print("Output folder:", animate_proc_dir)

Mode: replace
Character: /content/Wan2.2/madhuri.jpg
Source video: /content/Wan2.2/reel.mp4
Output folder: ./animate_proc_replace


In [10]:
import os

print("SAM2 config exists:",
      os.path.exists("configs/sam2/sam2_hiera_l.yaml"))

print("Working directory:", os.getcwd())

SAM2 config exists: False
Working directory: /content/Wan2.2


In [11]:
import sam2
import os

sam2_dir = os.path.dirname(sam2.__file__)

print("SAM2 package:", sam2_dir)
print("\nSearching for SAM2 configuration...")

for root, dirs, files in os.walk(sam2_dir):
    if "sam2_hiera_l.yaml" in files:
        print("FOUND:", os.path.join(root, "sam2_hiera_l.yaml"))

TypeError: expected str, bytes or os.PathLike object, not NoneType

In [12]:
import sam2
import importlib.util
import os

spec = importlib.util.find_spec("sam2")

print("SAM2 package location:", spec.submodule_search_locations)

sam2_dir = list(spec.submodule_search_locations)[0]

print("\nSearching for configuration...")

for root, dirs, files in os.walk(sam2_dir):
    if "sam2_hiera_l.yaml" in files:
        print("FOUND:", os.path.join(root, "sam2_hiera_l.yaml"))

SAM2 package location: _NamespacePath(['/content/sam2', '/content/Wan2.2/sam2'])

Searching for configuration...
FOUND: /content/sam2/sam2/sam2_hiera_l.yaml
FOUND: /content/sam2/sam2/configs/sam2/sam2_hiera_l.yaml


In [13]:
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

GlobalHydra.instance().clear()

with initialize_config_dir(
    version_base=None,
    config_dir="/content/sam2/sam2/configs"
):
    cfg = compose(config_name="sam2_hiera_l")

print("SAM2 configuration loaded successfully!")
print("Model:", cfg.model._target_)

MissingConfigException: Cannot find primary config 'sam2_hiera_l'. Check that it's in your config search path.

Config search path:
	provider=hydra, path=pkg://hydra.conf
	provider=main, path=file:///content/sam2/sam2/configs
	provider=schema, path=structured://

In [14]:
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

GlobalHydra.instance().clear()

with initialize_config_dir(
    version_base=None,
    config_dir="/content/sam2/sam2/configs/sam2"
):
    cfg = compose(config_name="sam2_hiera_l")

print("SAM2 configuration loaded successfully!")
print("Model:", cfg.model._target_)

SAM2 configuration loaded successfully!
Model: sam2.modeling.sam2_base.SAM2Base


In [15]:
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

GlobalHydra.instance().clear()

# Keep Hydra initialized for the next preprocessing cell
sam2_hydra_context = initialize_config_dir(
    version_base=None,
    config_dir="/content/sam2/sam2/configs/sam2"
)

sam2_hydra_context.__enter__()

# Test the exact filename used by Wan2.2
cfg = compose(config_name="sam2_hiera_l.yaml")

print("Wan2.2 SAM2 configuration:", cfg.model._target_)
print("Hydra configuration registered successfully!")

Wan2.2 SAM2 configuration: sam2.modeling.sam2_base.SAM2Base
Hydra configuration registered successfully!


In [16]:
import torch
from sam2.build_sam import build_sam2_video_predictor

checkpoint = (
    "./Wan2.2-Animate-14B/process_checkpoint/"
    "sam2/sam2_hiera_large.pt"
)

model = build_sam2_video_predictor(
    config_file="sam2_hiera_l.yaml",
    ckpt_path=checkpoint,
    device="cuda"
)

print("SAM2 model initialized successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("Allocated VRAM:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

del model
torch.cuda.empty_cache()

RuntimeError: You're likely running Python from the parent directory of the sam2 repository (i.e. the directory where https://github.com/facebookresearch/sam2 is cloned into). This is not supported since the `sam2` Python package could be shadowed by the repository name (the repository is also named `sam2` and contains the Python package in `sam2/sam2`). Please run Python from another directory (e.g. from the repo dir rather than its parent dir, or from your home directory) after installing SAM 2.

In [17]:
import os
import sys
import sam2

print("Current directory:", os.getcwd())
print("\nSAM2 package locations:")

for p in sam2.__path__:
    print(p, "| contains nested sam2 folder:",
          os.path.isdir(os.path.join(p, "sam2")))

print("\nPython search paths:")
for p in sys.path:
    if "sam2" in p.lower() or p == "/content":
        print(p)

Current directory: /content/Wan2.2

SAM2 package locations:
/content/sam2 | contains nested sam2 folder: True
/content/Wan2.2/sam2 | contains nested sam2 folder: True

Python search paths:
/content


In [18]:
import os
import sys
import importlib

# Move outside both SAM2 repository parent directories
os.chdir("/content")

# Remove conflicting paths
sys.path = [
    p for p in sys.path
    if os.path.abspath(p or "/content") != "/content"
]

# Re-import SAM2 from the installed package
for name in list(sys.modules):
    if name == "sam2" or name.startswith("sam2."):
        del sys.modules[name]

import sam2

print("Current directory:", os.getcwd())
print("SAM2 package:", list(sam2.__path__))

Current directory: /content
SAM2 package: ['/content/sam2/sam2']


In [19]:
import os
import sam2

for p in sam2.__path__:
    print(
        p,
        "Nested repository:",
        os.path.isdir(os.path.join(p, "sam2"))
    )

/content/sam2/sam2 Nested repository: False


In [20]:
import os
import torch

os.chdir("/content")

from sam2.build_sam import build_sam2_video_predictor

checkpoint = (
    "/content/Wan2.2/Wan2.2-Animate-14B/"
    "process_checkpoint/sam2/sam2_hiera_large.pt"
)

print("Checkpoint exists:", os.path.isfile(checkpoint))

model = build_sam2_video_predictor(
    config_file="sam2_hiera_l.yaml",
    ckpt_path=checkpoint,
    device="cuda"
)

print("SAM2 initialized successfully!")
print("VRAM allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

del model
torch.cuda.empty_cache()

Checkpoint exists: True
SAM2 initialized successfully!
VRAM allocated: 0.98 GB


In [21]:
import os

os.chdir("/content/Wan2.2")

print("Directory:", os.getcwd())
print("Character:", os.path.exists("madhuri.jpg"))
print("Video:", os.path.exists("reel.mp4"))

Directory: /content/Wan2.2
Character: True
Video: True


In [22]:
from hydra import initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

GlobalHydra.instance().clear()

sam2_hydra_context = initialize_config_dir(
    version_base=None,
    config_dir="/content/sam2/sam2/configs/sam2"
)

sam2_hydra_context.__enter__()

print("SAM2 configuration ready for Wan2.2")

SAM2 configuration ready for Wan2.2


In [23]:
import os

os.chdir("/content/Wan2.2")

print("Working directory:", os.getcwd())

for f in ["madhuri.jpg", "reel.mp4"]:
    print(f, "FOUND" if os.path.isfile(f) else "MISSING")

Working directory: /content/Wan2.2
madhuri.jpg FOUND
reel.mp4 FOUND


In [ ]:
# ── 6.2  Choose Mode & Resolution ────────────────────────────────────────────
# @title Animate Settings

animate_mode = "animate"  # @param ["animate", "replace"]

# @markdown Resolution in pixels: width height (must be divisible by 16)
animate_res_w = 1280  # @param {type:"integer"}
animate_res_h = 720   # @param {type:"integer"}

# Preprocessing output directory
animate_proc_dir = f"./animate_proc_{animate_mode}"
os.makedirs(animate_proc_dir, exist_ok=True)

print(f"Mode: {animate_mode}")
print(f"Resolution: {animate_res_w}x{animate_res_h}")
print(f"Preprocessing output: {animate_proc_dir}")

In [24]:
animate_mode = "replace"

animate_src_video = "/content/Wan2.2/reel.mp4"
animate_ref_image = "/content/Wan2.2/madhuri.jpg"

animate_res_w = 832
animate_res_h = 480

animate_proc_dir = "/content/Wan2.2/animate_proc_replace"

print("Mode:", animate_mode)
print("Video:", animate_src_video)
print("Reference:", animate_ref_image)
print("Output:", animate_proc_dir)

Mode: replace
Video: /content/Wan2.2/reel.mp4
Reference: /content/Wan2.2/madhuri.jpg
Output: /content/Wan2.2/animate_proc_replace


In [25]:
import os

script = "/content/Wan2.2/wan/modules/animate/preprocess/preprocess_data.py"

print("Script exists:", os.path.isfile(script))

with open(script, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if any(word in line.lower() for word in ["sam2", "hydra", "config_file"]):
        start = max(0, i - 2)
        end = min(len(lines), i + 4)

        print(f"\n--- Lines {start+1}-{end} ---")
        print("".join(lines[start:end]))

Script exists: True

--- Lines 103-108 ---
    det_checkpoint_path = os.path.join(args.ckpt_path, 'det/yolov10m.onnx')

    sam2_checkpoint_path = os.path.join(args.ckpt_path, 'sam2/sam2_hiera_large.pt') if args.replace_flag else None
    flux_kontext_path = os.path.join(args.ckpt_path, 'FLUX.1-Kontext-dev') if args.use_flux else None
    process_pipeline = ProcessPipeline(det_checkpoint_path=det_checkpoint_path, pose2d_checkpoint_path=pose2d_checkpoint_path, sam_checkpoint_path=sam2_checkpoint_path, flux_kontext_path=flux_kontext_path)
    os.makedirs(args.save_path, exist_ok=True)


--- Lines 105-110 ---
    sam2_checkpoint_path = os.path.join(args.ckpt_path, 'sam2/sam2_hiera_large.pt') if args.replace_flag else None
    flux_kontext_path = os.path.join(args.ckpt_path, 'FLUX.1-Kontext-dev') if args.use_flux else None
    process_pipeline = ProcessPipeline(det_checkpoint_path=det_checkpoint_path, pose2d_checkpoint_path=pose2d_checkpoint_path, sam_checkpoint_path=sam2_checkpoint_path, 

In [26]:
script = "/content/Wan2.2/wan/modules/animate/preprocess/preprocess_data.py"

with open(script, "r") as f:
    lines = f.readlines()

for i in range(85, 120):
    print(f"{i+1}: {lines[i]}", end="")

86:         default=1,
87:         help="The number of subdivisions for the grid along the 'h' dimension. A higher value results in a more detailed contour. A value of 1 means no subdivision is performed."
88:     )
89:     args = parser.parse_args()
90: 
91:     return args
92: 
93: 
94: if __name__ == '__main__':
95:     args = _parse_args()
96:     args_dict = vars(args)
97:     print(args_dict)
98: 
99:     assert len(args.resolution_area) == 2, "resolution_area should be a list of two integers [width, height]"
100:     assert not args.use_flux or args.retarget_flag, "Image editing with FLUX can only be used when pose retargeting is enabled."
101: 
102:     pose2d_checkpoint_path = os.path.join(args.ckpt_path, 'pose2d/vitpose_h_wholebody.onnx')
103:     det_checkpoint_path = os.path.join(args.ckpt_path, 'det/yolov10m.onnx')
104: 
105:     sam2_checkpoint_path = os.path.join(args.ckpt_path, 'sam2/sam2_hiera_large.pt') if args.replace_flag else None
106:     flux_kontext_path = os.pa

In [27]:
import os

root = "/content/Wan2.2/wan/modules/animate/preprocess"

for current, dirs, files in os.walk(root):
    for filename in files:
        if filename.endswith(".py"):
            path = os.path.join(current, filename)

            with open(path, "r") as f:
                content = f.read()

            if "class ProcessPipeline" in content:
                print("FOUND:", path)

                lines = content.splitlines()

                for i, line in enumerate(lines):
                    if "class ProcessPipeline" in line:
                        start = max(0, i - 3)
                        end = min(len(lines), i + 100)

                        for j in range(start, end):
                            print(f"{j+1}: {lines[j]}")
                        break

FOUND: /content/Wan2.2/wan/modules/animate/preprocess/process_pipepline.py
25: from sam_utils import build_sam2_video_predictor
26: 
27: 
28: class ProcessPipeline():
29:     def __init__(self, det_checkpoint_path, pose2d_checkpoint_path, sam_checkpoint_path, flux_kontext_path):
30:         self.pose2d = Pose2d(checkpoint=pose2d_checkpoint_path, detector_checkpoint=det_checkpoint_path)
31: 
32:         model_cfg = "sam2_hiera_l.yaml"
33:         if sam_checkpoint_path is not None:
34:             self.predictor = build_sam2_video_predictor(model_cfg, sam_checkpoint_path)
35:         if flux_kontext_path is not None:
36:             self.flux_kontext = FluxKontextPipeline.from_pretrained(flux_kontext_path, torch_dtype=torch.bfloat16).to("cuda")
37: 
38:     def __call__(self, video_path, refer_image_path, output_path, resolution_area=[1280, 720], fps=30, iterations=3, k=7, w_len=1, h_len=1, retarget_flag=False, use_flux=False, replace_flag=False):
39:         if replace_flag:
40: 
41:  

In [28]:
path = "/content/Wan2.2/wan/modules/animate/preprocess/sam_utils.py"

with open(path, "r") as f:
    source = f.read()

old = """    # Read config and init model
    cfg = compose(config_name=config_file, overrides=hydra_overrides)
    OmegaConf.resolve(cfg)
"""

new = """    # Read config and init model
    from hydra import initialize_config_dir
    from hydra.core.global_hydra import GlobalHydra

    GlobalHydra.instance().clear()

    with initialize_config_dir(
        version_base=None,
        config_dir="/content/sam2/sam2/configs/sam2",
    ):
        cfg = compose(config_name=config_file, overrides=hydra_overrides)

    OmegaConf.resolve(cfg)
"""

if old not in source:
    raise RuntimeError("Expected code block not found. File was not changed.")

with open(path, "w") as f:
    f.write(source.replace(old, new, 1))

print("SAM2 helper patched successfully!")

SAM2 helper patched successfully!


In [29]:
path = "/content/Wan2.2/wan/modules/animate/preprocess/sam_utils.py"

with open(path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "initialize_config_dir" in line:
        print("".join(lines[max(0, i-3):i+14]))
        break


    hydra_overrides.extend(hydra_overrides_extra)
    # Read config and init model
    from hydra import initialize_config_dir
    from hydra.core.global_hydra import GlobalHydra

    GlobalHydra.instance().clear()

    with initialize_config_dir(
        version_base=None,
        config_dir="/content/sam2/sam2/configs/sam2",
    ):
        cfg = compose(config_name=config_file, overrides=hydra_overrides)

    OmegaConf.resolve(cfg)
    model = instantiate(cfg.model, _recursive_=True)
    _load_checkpoint(model, ckpt_path)



In [30]:
import os

print("Current directory:", os.getcwd())

print("\nSAM2 folders:")
for path in [
    "/content/sam2/sam2",
    "/content/Wan2.2/sam2",
]:
    print(path, "EXISTS" if os.path.isdir(path) else "NOT FOUND")

print("\nPreprocessing helper:")
print(os.path.exists(
    "/content/Wan2.2/wan/modules/animate/preprocess/sam_utils.py"
))

Current directory: /content/Wan2.2

SAM2 folders:
/content/sam2/sam2 EXISTS
/content/Wan2.2/sam2 EXISTS

Preprocessing helper:
True


In [31]:
import subprocess

test = subprocess.run(
    [
        "python", "-c",
        "import sam2; print('Imported SAM2 from:', list(sam2.__path__))"
    ],
    cwd="/content/Wan2.2",
    capture_output=True,
    text=True
)

print("OUTPUT:")
print(test.stdout)

print("ERROR:")
print(test.stderr)

OUTPUT:
Imported SAM2 from: ['/content/Wan2.2/sam2']

ERROR:



In [34]:
import subprocess
import os

code = """
import sam2
import sam2.build_sam
print("SAM2 package paths:", list(sam2.__path__))
print("SAM2 package file:", getattr(sam2, "__file__", None))
print("build_sam file:", sam2.build_sam.__file__)
"""

test = subprocess.run(
    ["python", "-c", code],
    cwd="/content",
    env={**os.environ, "PYTHONPATH": "/content"},
    capture_output=True,
    text=True
)

print(test.stdout)
print(test.stderr)


Traceback (most recent call last):
  File "<string>", line 3, in <module>
    import sam2.build_sam
  File "/content/sam2/sam2/build_sam.py", line 25, in <module>
    raise RuntimeError(
    ...<6 lines>...
    )
RuntimeError: You're likely running Python from the parent directory of the sam2 repository (i.e. the directory where https://github.com/facebookresearch/sam2 is cloned into). This is not supported since the `sam2` Python package could be shadowed by the repository name (the repository is also named `sam2` and contains the Python package in `sam2/sam2`). Please run Python from another directory (e.g. from the repo dir rather than its parent dir, or from your home directory) after installing SAM 2.



In [35]:
import subprocess
import os

code = """
import sam2
import sam2.build_sam
print("SAM2 package paths:", list(sam2.__path__))
print("SAM2 package file:", getattr(sam2, "__file__", None))
print("build_sam file:", sam2.build_sam.__file__)
"""

test = subprocess.run(
    ["python", "-c", code],
    cwd="/content",
    env={**os.environ, "PYTHONPATH": "/content"},
    capture_output=True,
    text=True
)

print(test.stdout)
print(test.stderr)


Traceback (most recent call last):
  File "<string>", line 3, in <module>
    import sam2.build_sam
  File "/content/sam2/sam2/build_sam.py", line 25, in <module>
    raise RuntimeError(
    ...<6 lines>...
    )
RuntimeError: You're likely running Python from the parent directory of the sam2 repository (i.e. the directory where https://github.com/facebookresearch/sam2 is cloned into). This is not supported since the `sam2` Python package could be shadowed by the repository name (the repository is also named `sam2` and contains the Python package in `sam2/sam2`). Please run Python from another directory (e.g. from the repo dir rather than its parent dir, or from your home directory) after installing SAM 2.



In [36]:
import subprocess
import os

env = os.environ.copy()
env["PYTHONPATH"] = "/content/sam2"

test = subprocess.run(
    [
        "python", "-c",
        """
import sam2
print("SAM2 package:", sam2.__file__)
print("SAM2 paths:", list(sam2.__path__))

import sam2.build_sam
print("Build SAM file:", sam2.build_sam.__file__)
print("SAM2 import successful!")
"""
    ],
    cwd="/tmp",
    env=env,
    capture_output=True,
    text=True
)

print("OUTPUT:")
print(test.stdout)
print("ERROR:")
print(test.stderr)

OUTPUT:
SAM2 package: /content/sam2/sam2/__init__.py
SAM2 paths: ['/content/sam2/sam2']
Build SAM file: /content/sam2/sam2/build_sam.py
SAM2 import successful!

ERROR:



In [37]:
import os
import subprocess

env = os.environ.copy()
env["PYTHONPATH"] = "/content/sam2"

test = subprocess.run(
    [
        "python", "-c",
        """
import os, sys
os.chdir('/content/Wan2.2')
sys.path.insert(0, '/content/Wan2.2/wan/modules/animate/preprocess')
import sam2
import sam_utils
print("SAM2:", sam2.__file__)
print("SAM helper:", sam_utils.__file__)
print("Preprocessing imports successful!")
"""
    ],
    cwd="/tmp",
    env=env,
    capture_output=True,
    text=True
)

print("OUTPUT:")
print(test.stdout)
print("ERROR:")
print(test.stderr)

OUTPUT:
SAM2: /content/sam2/sam2/__init__.py
SAM helper: /content/Wan2.2/wan/modules/animate/preprocess/sam_utils.py
Preprocessing imports successful!

ERROR:



In [45]:
# ── 6.3  Preprocess Input Video ───────────────────────────────────────────────
# This step extracts poses, faces, and other signals needed by the model.
# --- 6.3 Preprocess Input Video ---
import os
import sys
import subprocess
import torch

torch.cuda.empty_cache()

print(f"Running preprocessing in '{animate_mode}' mode...")

# Use the correct SAM2 package and keep Wan2.2 available
env = os.environ.copy()
env["PYTHONPATH"] = "/content/sam2:/content/Wan2.2"

# Run outside the Wan2.2 folder to avoid the SAM2 import conflict
script = "/content/Wan2.2/wan/modules/animate/preprocess/preprocess_data.py"

cmd = [
    sys.executable,
    script,
    "--ckpt_path", "/content/Wan2.2/Wan2.2-Animate-14B/process_checkpoint",
    "--video_path", animate_src_video,
    "--refer_path", animate_ref_image,
    "--save_path", animate_proc_dir,
    "--resolution_area", str(animate_res_w), str(animate_res_h),
    "--iterations", "3",
    "--k", "7",
    "--w_len", "1",
    "--h_len", "1",
    "--replace_flag",
]

print("Video:", animate_src_video)
print("Character:", animate_ref_image)
print("Output:", animate_proc_dir)
print("Launching preprocessing...")

subprocess.run(
    cmd,
    cwd="/tmp",
    env=env,
    check=True
)

print("Preprocessing completed successfully!")

Running preprocessing in 'replace' mode...
Video: /content/Wan2.2/reel.mp4
Character: /content/Wan2.2/madhuri.jpg
Output: /content/Wan2.2/animate_proc_replace
Launching preprocessing...
Preprocessing completed successfully!


In [46]:
import os

folder = "/content/Wan2.2/animate_proc_replace"

print("Preprocessing output files:\n")

for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)
    if os.path.isfile(path):
        print(f"{name}  |  {os.path.getsize(path) / (1024**2):.2f} MB")

Preprocessing output files:

src_bg.mp4  |  0.11 MB
src_face.mp4  |  0.50 MB
src_mask.mp4  |  0.03 MB
src_pose.mp4  |  0.64 MB
src_ref.png  |  0.03 MB


In [39]:
import os
import shutil
import subprocess

# RAM
try:
    import psutil
    mem = psutil.virtual_memory()
    print("SYSTEM RAM")
    print("Total:", round(mem.total / 1024**3, 2), "GB")
    print("Available:", round(mem.available / 1024**3, 2), "GB")
except Exception as e:
    print("RAM check:", e)

# GPU
print("\nGPU MEMORY")
result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout if result.returncode == 0 else result.stderr)

# Video size and metadata
video = "/content/Wan2.2/reel.mp4"
print("\nVIDEO")
print("Exists:", os.path.exists(video))

if os.path.exists(video):
    print("File size:", round(os.path.getsize(video) / 1024**2, 2), "MB")

    try:
        import cv2
        cap = cv2.VideoCapture(video)
        print("Width:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
        print("Height:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
        print("FPS:", cap.get(cv2.CAP_PROP_FPS))
        print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        print("Duration:", round(
            cap.get(cv2.CAP_PROP_FRAME_COUNT) /
            max(cap.get(cv2.CAP_PROP_FPS), 0.001), 2
        ), "seconds")
        cap.release()
    except Exception as e:
        print("Video metadata error:", e)

print("\nOUTPUT FOLDER")
folder = "/content/Wan2.2/animate_proc_replace"
print("Exists:", os.path.exists(folder))
if os.path.exists(folder):
    print("Files:", os.listdir(folder)[:20])

SYSTEM RAM
Total: 12.67 GB
Available: 6.84 GB

GPU MEMORY
Tue Nov  3 11:48:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             30W /   70W |     121MiB /  15360MiB |      0%      Default |
|                                         |                        |              

In [40]:
path = "/content/Wan2.2/wan/modules/animate/preprocess/process_pipepline.py"

with open(path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "def get_mask" in line:
        print("Found get_mask at line:", i + 1)
        print("".join(lines[i:i+100]))
        break
else:
    print("get_mask definition not found")

Found get_mask at line: 280
    def get_mask(self, frames, th_step, kp2ds_all):
        frame_num = len(frames)
        if frame_num < th_step:
            num_step = 1
        else:
            num_step = (frame_num + th_step) // th_step

        all_mask = []
        for index in range(num_step):
            each_frames = frames[index * th_step:(index + 1) * th_step]
    
            kp2ds = kp2ds_all[index * th_step:(index + 1) * th_step]
            if len(each_frames) > 4:
                key_frame_num = 4
            elif 4 >= len(each_frames) > 0:
                key_frame_num = 1
            else:
                continue

            key_frame_step = len(kp2ds) // key_frame_num
            key_frame_index_list = list(range(0, len(kp2ds), key_frame_step))

            key_points_index = [0, 1, 2, 5, 8, 11, 10, 13]
            key_frame_body_points_list = []
            for key_frame_index in key_frame_index_list:
                keypoints_body_list = []
                body_key

In [41]:
path = "/content/Wan2.2/wan/modules/animate/preprocess/process_pipepline.py"

with open(path, "r") as f:
    lines = f.readlines()

for i in range(279, 390):
    print(f"{i+1}: {lines[i]}", end="")

280:     def get_mask(self, frames, th_step, kp2ds_all):
281:         frame_num = len(frames)
282:         if frame_num < th_step:
283:             num_step = 1
284:         else:
285:             num_step = (frame_num + th_step) // th_step
286: 
287:         all_mask = []
288:         for index in range(num_step):
289:             each_frames = frames[index * th_step:(index + 1) * th_step]
290:     
291:             kp2ds = kp2ds_all[index * th_step:(index + 1) * th_step]
292:             if len(each_frames) > 4:
293:                 key_frame_num = 4
294:             elif 4 >= len(each_frames) > 0:
295:                 key_frame_num = 1
296:             else:
297:                 continue
298: 
299:             key_frame_step = len(kp2ds) // key_frame_num
300:             key_frame_index_list = list(range(0, len(kp2ds), key_frame_step))
301: 
302:             key_points_index = [0, 1, 2, 5, 8, 11, 10, 13]
303:             key_frame_body_points_list = []
304:             for key_frame

IndexError: list index out of range

In [42]:
path = "/content/Wan2.2/wan/modules/animate/preprocess/process_pipepline.py"

with open(path, "r") as f:
    lines = f.readlines()

for i in range(330, min(390, len(lines))):
    print(f"{i+1}: {lines[i]}", end="")

331:             video_segments = {}
332:             for out_frame_idx, out_obj_ids, out_mask_logits in self.predictor.propagate_in_video(inference_state):
333:                 video_segments[out_frame_idx] = {
334:                     out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
335:                     for i, out_obj_id in enumerate(out_obj_ids)
336:                 }
337: 
338:             for out_frame_idx in range(len(video_segments)):
339:                 for out_obj_id, out_mask in video_segments[out_frame_idx].items():
340:                     out_mask = out_mask[0].astype(np.uint8)
341:                     all_mask.append(out_mask)
342: 
343:         return all_mask
344:     
345:     def convert_list_to_array(self, metas):
346:         metas_list = []
347:         for meta in metas:
348:             for key, value in meta.items():
349:                 if type(value) is list:
350:                     value = np.array(value)
351:                 meta[key] = value
352: 

In [43]:
path = "/content/Wan2.2/wan/modules/animate/preprocess/process_pipepline.py"

with open(path, "r") as f:
    source = f.read()

old = "masks = self.get_mask(frames, 400, tpl_pose_metas)"
new = "masks = self.get_mask(frames, 32, tpl_pose_metas)"

if source.count(old) != 1:
    raise RuntimeError("Expected line not found exactly once. No changes made.")

backup = path + ".backup"

with open(backup, "w") as f:
    f.write(source)

with open(path, "w") as f:
    f.write(source.replace(old, new, 1))

print("Batch size changed: 400 → 32")
print("Original file backed up:", backup)

Batch size changed: 400 → 32
Original file backed up: /content/Wan2.2/wan/modules/animate/preprocess/process_pipepline.py.backup


In [44]:
path = "/content/Wan2.2/wan/modules/animate/preprocess/process_pipepline.py"

with open(path, "r") as f:
    for line in f:
        if "self.get_mask(frames" in line:
            print(line.strip())

masks = self.get_mask(frames, 32, tpl_pose_metas)


In [ ]:
# ── 6.4a  Run Animate — Animation Mode ───────────────────────────────────────
import torch
torch.cuda.empty_cache()

if animate_mode == "animate":
    !python generate.py \
        --task animate-14B \
        --ckpt_dir ./Wan2.2-Animate-14B \
        --src_root_path {animate_proc_dir} \
        --refert_num 1
else:
    print("Mode is 'replace' — run cell 6.4b instead.")

In [ ]:
# ── 6.4b  Run Animate — Replacement Mode ─────────────────────────────────────
import torch
torch.cuda.empty_cache()

if animate_mode == "replace":
    !python generate.py \
        --task animate-14B \
        --ckpt_dir ./Wan2.2-Animate-14B \
        --src_root_path {animate_proc_dir} \
        --refert_num 1 \
        --replace_flag \
        --use_relighting_lora
else:
    print("Mode is 'animate' — run cell 6.4a instead.")

In [1]:
path = "/content/Wan2.2/generate.py"

with open(path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "add_argument" in line or "offload" in line.lower():
        print(f"{i+1}: {line}", end="")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Wan2.2/generate.py'

In [2]:
import os

root = "/content/Wan2.2"

print("Searching for generate.py...\n")

for current_root, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [
        ".git", "__pycache__", "node_modules"
    ]]

    if "generate.py" in files:
        print(os.path.join(current_root, "generate.py"))

print("\nRepository root files:")
print(os.listdir(root)[:50])

Searching for generate.py...


Repository root files:


FileNotFoundError: [Errno 2] No such file or directory: '/content/Wan2.2'

In [3]:
import os

print("Current directory:", os.getcwd())
print("\nContents of /content:")

if os.path.exists("/content"):
    print(os.listdir("/content"))
else:
    print("/content does not exist")

print("\nWan2.2 folder exists:",
      os.path.exists("/content/Wan2.2"))

print("Preprocessed output exists:",
      os.path.exists("/content/Wan2.2/animate_proc_replace"))

Current directory: /content

Contents of /content:
['.config', 'sample_data']

Wan2.2 folder exists: False
Preprocessed output exists: False


In [4]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


---
## Section 7 — Display & Download Output

Find, display, and download the most recently generated video.

In [ ]:
# ── 7.1  List all generated videos ────────────────────────────────────────────
import glob, os

mp4_files = sorted(glob.glob("**/*.mp4", recursive=True), key=os.path.getctime, reverse=True)
if mp4_files:
    print(f"Found {len(mp4_files)} video(s):")
    for i, f in enumerate(mp4_files):
        size_mb = os.path.getsize(f) / 1e6
        print(f"  [{i}] {f}  ({size_mb:.1f} MB)")
else:
    print("No .mp4 files found. Generate a video first.")

In [ ]:
# ── 7.2  Preview the latest video ─────────────────────────────────────────────
import glob, os
from IPython.display import Video, display

mp4_files = sorted(glob.glob("**/*.mp4", recursive=True), key=os.path.getctime, reverse=True)

preview_index = 0  # @param {type:"integer"}

if mp4_files and preview_index < len(mp4_files):
    chosen = mp4_files[preview_index]
    print(f"Previewing: {chosen}")
    display(Video(chosen, embed=True, width=854, height=480))
else:
    print("No video to display.")

In [ ]:
# ── 7.3  Download a video to your computer ────────────────────────────────────
import glob, os
from google.colab import files as colab_files

mp4_files = sorted(glob.glob("**/*.mp4", recursive=True), key=os.path.getctime, reverse=True)

download_index = 0  # @param {type:"integer"}

if mp4_files and download_index < len(mp4_files):
    chosen = mp4_files[download_index]
    print(f"Downloading: {chosen}")
    colab_files.download(chosen)
else:
    print("No video to download.")

In [1]:
import huggingface_hub
from huggingface_hub import HfApi

print("Hugging Face Hub:", huggingface_hub.__version__)

api = HfApi()
print("Hugging Face API: Working")

Hugging Face Hub: 1.33.0
Hugging Face API: Working


In [2]:
import os
from huggingface_hub import snapshot_download

# Wan2.2 repository location
repo_dir = "/content/Wan2.2"

# Confirm the repository exists
if not os.path.isdir(repo_dir):
    raise FileNotFoundError(
        "Wan2.2 repository not found. Tell me before continuing."
    )

# Set the working directory
os.chdir(repo_dir)

# Download only preprocessing checkpoints
model_dir = os.path.join(repo_dir, "Wan2.2-Animate-14B")

snapshot_download(
    repo_id="Wan-AI/Wan2.2-Animate-14B",
    local_dir=model_dir,
    allow_patterns=[
        "process_checkpoint/det/yolov10m.onnx",
        "process_checkpoint/pose2d/vitpose_h_wholebody.onnx/**",
        "process_checkpoint/sam2/sam2_hiera_large.pt"
    ]
)

print("Preprocessing checkpoint download completed!")
print("Model directory:", model_dir)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 396 files:   0%|          | 0/396 [00:00<?, ?it/s]

Preprocessing checkpoint download completed!
Model directory: /content/Wan2.2/Wan2.2-Animate-14B


In [3]:
import os

base = "./Wan2.2-Animate-14B/process_checkpoint"

required = [
    "det/yolov10m.onnx",
    "pose2d/vitpose_h_wholebody.onnx/end2end.onnx",
    "sam2/sam2_hiera_large.pt"
]

for f in required:
    path = os.path.join(base, f)
    print(
        "FOUND" if os.path.isfile(path) else "MISSING",
        f
    )

FOUND det/yolov10m.onnx
FOUND pose2d/vitpose_h_wholebody.onnx/end2end.onnx
FOUND sam2/sam2_hiera_large.pt


In [4]:
import os
import onnxruntime as ort

base = "./Wan2.2-Animate-14B/process_checkpoint"

models = [
    os.path.join(base, "det/yolov10m.onnx"),
    os.path.join(base, "pose2d/vitpose_h_wholebody.onnx")
]

for model in models:
    print("\nTesting:", model)

    session = ort.InferenceSession(
        model,
        providers=["CPUExecutionProvider"]
    )

    print("Loaded successfully!")
    print("Input:", session.get_inputs()[0].shape)
    print("Output:", session.get_outputs()[0].shape)


Testing: ./Wan2.2-Animate-14B/process_checkpoint/det/yolov10m.onnx
Loaded successfully!
Input: [1, 3, 640, 640]
Output: [1, 300, 6]

Testing: ./Wan2.2-Animate-14B/process_checkpoint/pose2d/vitpose_h_wholebody.onnx


InvalidProtobuf: [ONNXRuntimeError] : 7 : INVALID_PROTOBUF : Load model from ./Wan2.2-Animate-14B/process_checkpoint/pose2d/vitpose_h_wholebody.onnx failed:Protobuf parsing failed.

In [5]:
import os
import onnxruntime as ort

vitpose_dir = (
    "./Wan2.2-Animate-14B/process_checkpoint/"
    "pose2d/vitpose_h_wholebody.onnx"
)

vitpose_file = os.path.join(vitpose_dir, "end2end.onnx")

print("Model exists:", os.path.isfile(vitpose_file))

session = ort.InferenceSession(
    vitpose_file,
    providers=["CPUExecutionProvider"]
)

print("VitPose loaded successfully!")
print("Input:", session.get_inputs()[0].shape)
print("Output:", session.get_outputs()[0].shape)

Model exists: True
VitPose loaded successfully!
Input: ['batch_size', 3, 256, 192]
Output: ['batch_size', 133, 'Convoutput_0_dim_2', 'Convoutput_0_dim_3']


In [6]:
import os
import torch

sam_checkpoint = (
    "./Wan2.2-Animate-14B/process_checkpoint/"
    "sam2/sam2_hiera_large.pt"
)

print("SAM2 checkpoint exists:", os.path.isfile(sam_checkpoint))
print("PyTorch CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

SAM2 checkpoint exists: True
PyTorch CUDA available: True
GPU: Tesla T4
